In [1]:
%uv pip install ipykernel jupyterlab

Using Python 3.10.13 environment at: /usr/local
Resolved 93 packages in 433ms
⠙ Preparing packages... (0/47)
⠙ Preparing packages... (0/47)
⠙ Preparing packages... (0/47)
⠙ Preparing packages... (0/47)
notebook-shim        ------------------------------ 13.00 KiB/13.00 KiB
⠙ Preparing packages... (0/47)
notebook-shim        ------------------------------ 13.00 KiB/13.00 KiB
⠙ Preparing packages... (0/47)
notebook-shim        ------------------------------ 13.00 KiB/13.00 KiB
⠙ Preparing packages... (0/47)
notebook-shim        ------------------------------ 13.00 KiB/13.00 KiB
python-json-logger   ------------------------------     0 B/15.18 KiB
⠙ Preparing packages... (0/47)
notebook-shim        ------------------------------ 13.00 KiB/13.00 KiB
python-json-logger   ------------------------------     0 B/15.18 KiB
⠙ Preparing packages... (0/47)
notebook-shim        ------------------------------ 13.00 KiB/13.00 KiB
python-json-logger   ------------------------------     0 B/15.18 KiB
⠙

In [1]:
import torch
import torch.nn as nn

try:
    from mamba_ssm import Mamba
except ImportError:
    raise ImportError("Please install mamba_ssm and causal-conv1d to use this model.")


class IOHMambaPredictor(nn.Module):
    def __init__(self, input_dim=4, model_dim_1=32, model_dim_2=64):
        super(IOHMambaPredictor, self).__init__()

        self.input_projection_1 = nn.Linear(input_dim, model_dim_1)
        self.norm_1 = nn.LayerNorm(model_dim_1)
        self.mamba_1 = Mamba(d_model=model_dim_1, d_state=16, d_conv=4, expand=2)

        self.input_projection_2 = nn.Linear(model_dim_1, model_dim_2)
        self.norm_2 = nn.LayerNorm(model_dim_2)
        self.mamba_2 = Mamba(d_model=model_dim_2, d_state=16, d_conv=4, expand=2)

        self.output_projection = nn.Sequential(
            nn.Linear(model_dim_2 + 5, model_dim_2 // 2),
            nn.ELU(),
            nn.Linear(model_dim_2 // 2, 1)
        )

    # ── Training / evaluation on full windows (parallel scan, unchanged) ──────
    def forward(self, x_seq, x_static):
        x = self.input_projection_1(x_seq)  # (B, 900, 32)
        x = self.norm_1(x)
        x = self.mamba_1(x)                 # (B, 900, 32)

        x = self.input_projection_2(x)      # (B, 900, 64)
        x = self.norm_2(x)
        x = self.mamba_2(x)                 # (B, 900, 64)

        x = torch.mean(x, dim=1)            # (B, 64)
        x = torch.cat([x, x_static], dim=-1)
        return self.output_projection(x).squeeze(-1)  # (B,)

    # ── Recurrent inference: allocate states once per patient ────────────────
    def init_states(self, batch_size: int, device: torch.device):
        """
        Call once before processing a new patient.
        Returns two (conv_state, ssm_state) tuples, one per Mamba block.
        Pass these into step() and store the returned states each call.
        """
        d_inner_1 = self.mamba_1.d_inner   # model_dim_1 * expand = 64
        d_inner_2 = self.mamba_2.d_inner   # model_dim_2 * expand = 128

        state_1 = (
            torch.zeros(batch_size, d_inner_1, self.mamba_1.d_conv - 1, device=device),  # conv
            torch.zeros(batch_size, d_inner_1, self.mamba_1.d_state,    device=device),  # ssm
        )
        state_2 = (
            torch.zeros(batch_size, d_inner_2, self.mamba_2.d_conv - 1, device=device),
            torch.zeros(batch_size, d_inner_2, self.mamba_2.d_state,    device=device),
        )
        return state_1, state_2

    # ── Single recurrent step: O(1), no window buffer ────────────────────────
    def step(self, x_t, x_static, state_1, state_2):
        """
        x_t      : (B, 4)
        x_static : (B, 5)
        """
        conv_state_1, ssm_state_1 = state_1
        conv_state_2, ssm_state_2 = state_2
    
        # Block 1
        x = self.input_projection_1(x_t)    # (B, 32)
        x = self.norm_1(x)
        x = x.unsqueeze(1)                  # (B, 1, 32)  ← mamba_ssm.step() requires this
        x, conv_state_1, ssm_state_1 = self.mamba_1.step(x, conv_state_1, ssm_state_1)
        x = x.squeeze(1)                    # (B, 32)
    
        # Block 2
        x = self.input_projection_2(x)      # (B, 64)
        x = self.norm_2(x)
        x = x.unsqueeze(1)                  # (B, 1, 64)
        x, conv_state_2, ssm_state_2 = self.mamba_2.step(x, conv_state_2, ssm_state_2)
        x = x.squeeze(1)                    # (B, 64)
    
        # Output
        x = torch.cat([x, x_static], dim=-1)
        logit = self.output_projection(x).squeeze(-1)   # (B,)
    
        return logit, (conv_state_1, ssm_state_1), (conv_state_2, ssm_state_2)

/usr/local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import math

import torch
import torch.nn as nn

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int = 32, max_len: int = 90000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len, :]
        
        return x

class IOHTransformer(nn.Module):
    def __init__(self, model_dim=64, num_heads=8):
        super(IOHTransformer, self).__init__()
        self.multihead_attn = nn.MultiheadAttention(embed_dim=model_dim, num_heads=num_heads, batch_first=True)
        self.layer_norm_attn = nn.LayerNorm(model_dim)

        self.ffn = nn.Sequential(
            nn.Linear(model_dim, model_dim * 4),
            nn.ReLU(),
            nn.Linear(model_dim * 4, model_dim)
        )
        self.layer_norm_ffn = nn.LayerNorm(model_dim)

    def forward(self, x):
        # x shape: (Batch, seq_len, input_dim)
        attn_output, _ = self.multihead_attn(x, x, x)  # (Batch, seq_len, model_dim)
        x = self.layer_norm_attn(x + attn_output)  # Residual connection + LayerNorm

        ffn_output = self.ffn(x)  # (Batch, seq_len, model_dim)
        x = self.layer_norm_ffn(x + ffn_output)  # Residual connection + LayerNorm

        return x

class IOHPredictor(nn.Module):
    def __init__(self, input_dim = 4, model_dim_1 = 32, model_dim_2 = 64, num_heads = 4):
        super(IOHPredictor, self).__init__()
        self.input_projection_1 = nn.Linear(input_dim, model_dim_1)
        self.pos_embed = PositionalEncoding(d_model=model_dim_1)
        self.transformer_1 = IOHTransformer(model_dim=model_dim_1, num_heads=num_heads)

        self.conv1d = nn.Conv1d(model_dim_1, model_dim_2, kernel_size=5, stride=2)

        # self.input_projection_2 = nn.Linear(model_dim_1, model_dim_2)
        self.transformer_2 = IOHTransformer(model_dim=model_dim_2, num_heads=num_heads)
        self.output_projection = nn.Sequential(
            nn.Linear(model_dim_2+5, model_dim_2 // 2),
            nn.GELU(),
            nn.Linear(model_dim_2 // 2, 1)
        )
    
    def forward(self, x_seq, x_static):
        x = self.input_projection_1(x_seq) # (B, 900, 4)
        x = self.pos_embed(x)  # (B, 900, 4) -> (B, 900, model_dim)
        x = self.transformer_1(x)  # (Batch, 900, 32)

        # x = self.input_projection_2(x)
        x = self.conv1d(x.transpose(1, 2)).transpose(1, 2)
        x = torch.nn.functional.gelu(x)
        x = self.transformer_2(x)  # (Batch, 900, 64)

        x = torch.mean(x, dim=1)  # (B, 64)
        x = torch.concat([x, x_static], dim=-1)  # (B, 64 + 5)

        output = self.output_projection(x)  # (B, 1)
        return output.squeeze(-1)  # (B,)

if __name__ == "__main__":
    model = IOHPredictor(input_dim=4, model_dim_1=32, model_dim_2=64, num_heads=4)
    dummy_seq = torch.randn(2, 900, 4)  # (Batch, seq_len, input_dim)
    dummy_static = torch.randn(2, 5)# (Batch, static_dim)

    print("Parameters: ", sum(p.numel() for p in model.parameters()))

    output = model(dummy_seq, dummy_static)
    print("Output shape:", output.shape)

Parameters:  75425
Output shape: torch.Size([2])


In [5]:
"""
benchmark.py
============
Measures inference latency and peak VRAM for:
  - IOHMambaPredictor  : single recurrent step  → input (1, 4)
  - IOHPredictor       : full window recompute  → input (1, 900, 4)

Run with:
    python benchmark.py

Requires CUDA. Will raise clearly if CUDA is unavailable.
"""

import time
import torch

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
WARMUP_ITERS   = 50
MEASURE_ITERS  = 1000
BATCH_SIZE     = 1       # deployment processes one patient at a time
SEQ_LEN        = 500     # full observation window (Transformer)
INPUT_DIM      = 4
STATIC_DIM     = 5


# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def reset_vram_stats(device):
    """Zero out the peak-memory counter so measurements are isolated."""
    torch.cuda.reset_peak_memory_stats(device)
    torch.cuda.synchronize(device)


def peak_vram_mb(device):
    """Return peak allocated VRAM in MB since the last reset."""
    return torch.cuda.max_memory_allocated(device) / 1024 ** 2


def measure_latency(fn, warmup, iters, device):
    """
    Run fn() warmup times (discarded), then time iters calls.
    Returns mean latency in milliseconds.
    """
    for _ in range(warmup):
        fn()

    torch.cuda.synchronize(device)
    t0 = time.perf_counter()
    for _ in range(iters):
        fn()
    torch.cuda.synchronize(device)
    elapsed = time.perf_counter() - t0

    return elapsed / iters * 1000   # ms per call


# ─────────────────────────────────────────────────────────────────────────────
# BENCHMARK: Transformer — full window recomputation
# ─────────────────────────────────────────────────────────────────────────────
def bench_transformer(device):
    model = IOHPredictor(
        input_dim=INPUT_DIM,
        model_dim_1=32,
        model_dim_2=64,
        num_heads=4
    ).to(device).eval()

    dummy_seq    = torch.randn(BATCH_SIZE, SEQ_LEN, INPUT_DIM, device=device)
    dummy_static = torch.randn(BATCH_SIZE, STATIC_DIM,         device=device)

    def fn():
        with torch.no_grad():
            _ = model(dummy_seq, dummy_static)

    # Peak VRAM: measured over warmup + measure to capture worst-case
    reset_vram_stats(device)
    latency_ms = measure_latency(fn, WARMUP_ITERS, MEASURE_ITERS, device)
    vram_mb    = peak_vram_mb(device)

    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return latency_ms, vram_mb, params


# ─────────────────────────────────────────────────────────────────────────────
# BENCHMARK: Mamba — single recurrent step (O(1) per new sample)
# ─────────────────────────────────────────────────────────────────────────────
def bench_mamba(device):
    model = IOHMambaPredictor(
        input_dim=INPUT_DIM,
        model_dim_1=32,
        model_dim_2=64
    ).to(device).eval()

    # (B, INPUT_DIM) — no sequence dimension in recurrent mode
    dummy_step   = torch.randn(BATCH_SIZE, INPUT_DIM,  device=device)
    dummy_static = torch.randn(BATCH_SIZE, STATIC_DIM, device=device)

    # States must persist across calls — a fresh state models the start of a
    # new patient. We reuse the same state object across warmup and measurement
    # so that the hidden state is warm (realistic deployment condition).
    state_1, state_2 = model.init_states(BATCH_SIZE, device)

    def fn():
        nonlocal state_1, state_2
        with torch.no_grad():
            _, state_1, state_2 = model.step(
                dummy_step, dummy_static, state_1, state_2
            )

    reset_vram_stats(device)
    latency_ms = measure_latency(fn, WARMUP_ITERS, MEASURE_ITERS, device)
    vram_mb    = peak_vram_mb(device)

    params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return latency_ms, vram_mb, params


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
def main():
    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is required for this benchmark. "
            "Mamba's custom kernels do not run on CPU."
        )

    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(device)
    budget_ms = 2000.0   # 0.5 Hz → one new prediction every 2 seconds

    print("=" * 65)
    print("  IOH Model Inference Benchmark")
    print(f"  GPU            : {gpu_name}")
    print(f"  Batch size     : {BATCH_SIZE}  (deployment: one patient)")
    print(f"  Warmup iters   : {WARMUP_ITERS}")
    print(f"  Measure iters  : {MEASURE_ITERS}")
    print(f"  Real-time budget (0.5 Hz): {budget_ms:.0f} ms per step")
    print("=" * 65)

    # ── Transformer ──────────────────────────────────────────────────────────
    print("\n[1/2] Benchmarking Transformer (full window, input [1, 900, 4]) …")
    t_lat, t_vram, t_params = bench_transformer(device)
    t_headroom = budget_ms - t_lat

    print(f"  Parameters         : {t_params:,}")
    print(f"  Inference latency  : {t_lat:.3f} ms")
    print(f"  Peak VRAM          : {t_vram:.2f} MB")
    print(f"  Headroom vs budget : {t_headroom:.1f} ms")

    # ── Mamba ─────────────────────────────────────────────────────────────────
    print("\n[2/2] Benchmarking Mamba (recurrent step, input [1, 4]) …")
    m_lat, m_vram, m_params = bench_mamba(device)
    m_headroom = budget_ms - m_lat

    print(f"  Parameters         : {m_params:,}")
    print(f"  Inference latency  : {m_lat:.3f} ms")
    print(f"  Peak VRAM          : {m_vram:.2f} MB")
    print(f"  Headroom vs budget : {m_headroom:.1f} ms")

    # ── Comparison ───────────────────────────────────────────────────────────
    speedup       = t_lat  / m_lat
    vram_reduction = (1 - m_vram / t_vram) * 100

    print("\n" + "=" * 65)
    print("  SUMMARY")
    print("=" * 65)
    print(f"  {'Metric':<30} {'Transformer':>12} {'Mamba':>12}")
    print(f"  {'-'*54}")
    print(f"  {'Parameters':<30} {t_params:>12,} {m_params:>12,}")
    print(f"  {'Inference latency (ms)':<30} {t_lat:>12.3f} {m_lat:>12.3f}")
    print(f"  {'Peak VRAM (MB)':<30} {t_vram:>12.2f} {m_vram:>12.2f}")
    print(f"  {'Headroom at 0.5 Hz (ms)':<30} {t_headroom:>12.1f} {m_headroom:>12.1f}")
    print(f"  {'-'*54}")
    print(f"  Mamba latency speedup      : {speedup:.1f}×")
    print(f"  Mamba VRAM reduction       : {vram_reduction:.1f}%")
    print("=" * 65)


if __name__ == "__main__":
    main()

  IOH Model Inference Benchmark
  GPU            : NVIDIA L4
  Batch size     : 1  (deployment: one patient)
  Warmup iters   : 50
  Measure iters  : 1000
  Real-time budget (0.5 Hz): 2000 ms per step

[1/2] Benchmarking Transformer (full window, input [1, 900, 4]) …
  Parameters         : 75,425
  Inference latency  : 1.573 ms
  Peak VRAM          : 26.38 MB
  Headroom vs budget : 1998.4 ms

[2/2] Benchmarking Mamba (recurrent step, input [1, 4]) …
  Parameters         : 47,297
  Inference latency  : 1.740 ms
  Peak VRAM          : 9.34 MB
  Headroom vs budget : 1998.3 ms

  SUMMARY
  Metric                          Transformer        Mamba
  ------------------------------------------------------
  Parameters                           75,425       47,297
  Inference latency (ms)                1.573        1.740
  Peak VRAM (MB)                        26.38         9.34
  Headroom at 0.5 Hz (ms)              1998.4       1998.3
  ------------------------------------------------------
